# 15 · Conformal Prediction

## What this notebook covers
Conformal prediction (CP) wraps any trained model and produces prediction sets (classification) or intervals (regression) with a *guaranteed* coverage probability — without distributional assumptions.

## Why it matters
Standard prediction intervals from quantile regression or Bayesian methods give *approximate* coverage that depends on model assumptions. Conformal prediction gives *exact* marginal coverage: if you ask for 90% intervals, exactly ≥90% of test points will be covered, regardless of the true data distribution.

## The split conformal procedure
1. Split data into train / calibration / test
2. Train model on train set
3. Compute *nonconformity scores* on calibration set: e.g., for regression, score = |y - ŷ|
4. Find the (1-α) quantile of calibration scores → q̂
5. At test time, output interval [ŷ - q̂, ŷ + q̂]

This is exact (up to finite-sample correction) and requires no assumptions beyond exchangeability.

## Adaptive conformal (RAPS for classification)
For classification, the naive CP outputs prediction *sets* — possibly multiple classes. Regularised Adaptive Prediction Sets (RAPS) makes sets smaller on average while maintaining coverage, by penalising large sets during calibration.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

print("Conformal prediction: guaranteed coverage without distributional assumptions")

In [ ]:
# ── Regression conformal ───────────────────────────────────────────────────────
X_r, y_r = make_regression(n_samples=2000, n_features=20, noise=30, random_state=0)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_r, y_r, test_size=0.4, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0)

reg = GradientBoostingRegressor(n_estimators=150, random_state=0)
reg.fit(X_tr, y_tr)

# Calibration nonconformity scores
cal_scores = np.abs(y_cal - reg.predict(X_cal))

def conformal_interval(model, X_test, cal_scores, alpha=0.10):
    """Split conformal prediction interval with (1-alpha) coverage guarantee."""
    n = len(cal_scores)
    # Finite-sample corrected quantile level
    q_level = np.ceil((1 - alpha) * (n + 1)) / n
    q_level = min(q_level, 1.0)
    q_hat = np.quantile(cal_scores, q_level)
    y_pred = model.predict(X_test)
    return y_pred - q_hat, y_pred + q_hat, q_hat

lo, hi, q_hat = conformal_interval(reg, X_te, cal_scores, alpha=0.10)
coverage = np.mean((y_te >= lo) & (y_te <= hi))
avg_width = np.mean(hi - lo)

print(f"Target coverage: 90%")
print(f"Achieved coverage: {coverage:.2%}")
print(f"q̂ (half-width): {q_hat:.2f}")
print(f"Average interval width: {avg_width:.2f}")

In [ ]:
# Coverage at multiple alpha levels
alphas = [0.05, 0.10, 0.20, 0.30]
rows = []
for alpha in alphas:
    lo_a, hi_a, q_a = conformal_interval(reg, X_te, cal_scores, alpha=alpha)
    cov = np.mean((y_te >= lo_a) & (y_te <= hi_a))
    rows.append({
        "Target coverage": f"{1-alpha:.0%}",
        "Achieved coverage": f"{cov:.2%}",
        "q̂ (half-width)": round(q_a, 2),
        "Avg width": round(np.mean(hi_a - lo_a), 2),
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ── Classification conformal (prediction sets) ─────────────────────────────────
X_c, y_c = make_classification(n_samples=2000, n_features=20, n_informative=10, n_classes=4,
                                n_clusters_per_class=1, random_state=0)
X_tr_c, X_tmp_c, y_tr_c, y_tmp_c = train_test_split(X_c, y_c, test_size=0.4, random_state=0)
X_cal_c, X_te_c, y_cal_c, y_te_c = train_test_split(X_tmp_c, y_tmp_c, test_size=0.5, random_state=0)

clf_c = GradientBoostingClassifier(n_estimators=150, random_state=0)
clf_c.fit(X_tr_c, y_tr_c)

# Nonconformity score: 1 - P(true class)
cal_proba = clf_c.predict_proba(X_cal_c)
cal_scores_c = 1 - cal_proba[np.arange(len(y_cal_c)), y_cal_c]

def conformal_prediction_set(clf, X_test, cal_scores, alpha=0.10):
    """Return prediction sets with (1-alpha) coverage."""
    n = len(cal_scores)
    q_level = np.ceil((1 - alpha) * (n + 1)) / n
    q_level = min(q_level, 1.0)
    q_hat = np.quantile(cal_scores, q_level)
    proba = clf.predict_proba(X_test)
    # Include all classes where 1 - P(class) <= q_hat
    sets = [list(np.where(1 - row <= q_hat)[0]) for row in proba]
    return sets, q_hat

pred_sets, q_hat_c = conformal_prediction_set(clf_c, X_te_c, cal_scores_c, alpha=0.10)
coverage_c = np.mean([y_te_c[i] in pred_sets[i] for i in range(len(y_te_c))])
avg_set_size = np.mean([len(s) for s in pred_sets])

print(f"Target coverage: 90%")
print(f"Achieved coverage: {coverage_c:.2%}")
print(f"Average prediction set size: {avg_set_size:.2f} (out of 4 classes)")
print(f"\nExample prediction sets (first 5):")
for i in range(5):
    print(f"  True: {y_te_c[i]}  Set: {pred_sets[i]}  Covered: {y_te_c[i] in pred_sets[i]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Regression: interval visualisation on 80 sorted test points
sort_idx = np.argsort(y_te)[:80]
x_plot = np.arange(80)
axes[0].fill_between(x_plot, lo[sort_idx], hi[sort_idx], alpha=0.3, color="steelblue", label=f"90% PI (cov={coverage:.0%})")
axes[0].scatter(x_plot, y_te[sort_idx], s=8, color="red", zorder=3, label="Actual")
axes[0].plot(x_plot, reg.predict(X_te)[sort_idx], color="steelblue", lw=1.5, label="Point pred")
axes[0].set_title("Conformal regression intervals")
axes[0].legend(fontsize=8)

# Classification: set size distribution
set_sizes = [len(s) for s in pred_sets]
axes[1].hist(set_sizes, bins=range(1, 6), align="left", rwidth=0.7, color="steelblue", alpha=0.8)
axes[1].set_xlabel("Prediction set size")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Classification set sizes  (avg={avg_set_size:.2f}, cov={coverage_c:.0%})")
axes[1].set_xticks(range(1, 5))

plt.tight_layout()
plt.savefig(f"{base}/15_conformal_prediction.png", dpi=120)
plt.show()